# Insurance Claims EDA on Processed Data

This notebook builds a processed, model-ready version of the dataset and performs EDA on that processed data.

What this notebook covers:
- raw data loading and feature engineering
- binary mapping and one-hot encoding
- processed data quality checks
- class imbalance inspection
- summary statistics and outlier audit on processed continuous features
- top correlations with `claim_status`
- class-wise mean difference analysis for processed continuous and encoded binary features
- distribution plots and PCA projection for processed data

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 6)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [ ]:
def add_engineered_features(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()

    if "Unnamed: 0" in frame.columns:
        frame = frame.drop(columns=["Unnamed: 0"])

    frame["torque_nm"] = frame["max_torque"].str.extract(r"(\d+\.?\d*)Nm")[0].astype(float)
    frame["torque_rpm"] = frame["max_torque"].str.extract(r"@(\d+)rpm")[0].astype(float)
    frame["power_bhp"] = frame["max_power"].str.extract(r"(\d+\.?\d*)bhp")[0].astype(float)
    frame["power_rpm"] = frame["max_power"].str.extract(r"@(\d+)rpm")[0].astype(float)

    frame["power_to_weight"] = frame["power_bhp"] / frame["gross_weight"]
    frame["torque_per_liter"] = frame["torque_nm"] / (frame["displacement"] / 1000)
    frame["rpm_diff"] = frame["power_rpm"] - frame["torque_rpm"]

    frame["vehicle_age_bucket"] = pd.cut(
        frame["vehicle_age"],
        bins=[-0.01, 1, 3, 5, np.inf],
        labels=["0-1y", "1-3y", "3-5y", "5y+"],
    )
    frame["customer_age_bucket"] = pd.cut(
        frame["customer_age"],
        bins=[34, 40, 45, 50, 55, np.inf],
        labels=["35-40", "41-45", "46-50", "51-55", "56+"],
        include_lowest=True,
    )
    frame["subscription_bucket"] = pd.cut(
        frame["subscription_length"],
        bins=[-0.01, 2, 5, 10, np.inf],
        labels=["<2y", "2-5y", "5-10y", "10y+"],
    )

    frame = frame.drop(columns=["max_torque", "max_power"])
    return frame


def build_processed_dataset(frame: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    frame = add_engineered_features(frame)

    binary_cols = [
        "is_esc",
        "is_adjustable_steering",
        "is_tpms",
        "is_parking_sensors",
        "is_parking_camera",
        "is_front_fog_lights",
        "is_rear_window_wiper",
        "is_rear_window_washer",
        "is_rear_window_defogger",
        "is_brake_assist",
        "is_power_door_locks",
        "is_central_locking",
        "is_power_steering",
        "is_driver_seat_height_adjustable",
        "is_day_night_rear_view_mirror",
        "is_ecw",
        "is_speed_alert",
    ]

    for col in binary_cols:
        frame[col] = frame[col].map({"Yes": 1, "No": 0}).astype(int)

    categorical_cols = frame.select_dtypes(include=["object", "category", "string"]).columns.tolist()
    processed = pd.get_dummies(frame, columns=categorical_cols, drop_first=True, dtype=int)
    return processed, binary_cols


def top_correlations(frame: pd.DataFrame, target: str = "claim_status", top_n: int = 15) -> pd.DataFrame:
    corr_series = frame.corr(numeric_only=True)[target].drop(target)
    corr_series = corr_series.sort_values(key=lambda s: s.abs(), ascending=False).head(top_n)
    return corr_series.rename("correlation").reset_index().rename(columns={"index": "feature"})


def iqr_outlier_audit(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    rows = []
    for col in columns:
        q1 = frame[col].quantile(0.25)
        q3 = frame[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        outlier_count = int(((frame[col] < lower) | (frame[col] > upper)).sum())
        rows.append(
            {
                "feature": col,
                "outlier_count": outlier_count,
                "outlier_pct": round(outlier_count / len(frame) * 100, 2),
                "lower_bound": round(lower, 4),
                "upper_bound": round(upper, 4),
            }
        )
    return pd.DataFrame(rows).sort_values("outlier_pct", ascending=False)

In [ ]:
data_path = Path("Insurance claims data.csv")
df_raw = pd.read_csv(data_path)
df_engineered = add_engineered_features(df_raw)
df_processed, original_binary_cols = build_processed_dataset(df_raw)

feature_cols = [col for col in df_processed.columns if col != "claim_status"]
binary_processed_cols = [
    col for col in feature_cols if set(pd.Series(df_processed[col]).dropna().unique()).issubset({0, 1})
]
continuous_processed_cols = [col for col in feature_cols if col not in binary_processed_cols]

print(f"Raw shape: {df_raw.shape}")
print(f"Engineered shape: {df_engineered.shape}")
print(f"Processed shape: {df_processed.shape}")
print(f"Processed binary / dummy features: {len(binary_processed_cols)}")
print(f"Processed continuous features: {len(continuous_processed_cols)}")
df_processed.head()

In [ ]:
quality_summary = pd.DataFrame(
    {
        "metric": [
            "raw_missing_values",
            "raw_full_row_duplicates",
            "engineered_feature_duplicates_ex_target",
            "processed_missing_values",
            "processed_feature_duplicates_ex_target",
            "processed_total_features",
            "processed_binary_dummy_features",
            "processed_continuous_features",
        ],
        "value": [
            int(df_raw.isna().sum().sum()),
            int(df_raw.duplicated().sum()),
            int(df_engineered.drop(columns=["claim_status"]).duplicated().sum()),
            int(df_processed.isna().sum().sum()),
            int(df_processed.drop(columns=["claim_status"]).duplicated().sum()),
            len(feature_cols),
            len(binary_processed_cols),
            len(continuous_processed_cols),
        ],
    }
)

print("Processed data quality summary")
print(quality_summary.to_string(index=False))

processed_dtypes = df_processed.dtypes.astype(str).value_counts().rename_axis("dtype").reset_index(name="count")
print("\nProcessed dtype distribution")
print(processed_dtypes.to_string(index=False))

In [ ]:
target_dist = df_processed["claim_status"].value_counts().sort_index().to_frame("count")
target_dist["share_pct"] = (target_dist["count"] / len(df_processed) * 100).round(2)
imbalance_ratio = round(target_dist.loc[0, "count"] / target_dist.loc[1, "count"], 2)

print(target_dist.to_string())
print(f"\nMajority to minority ratio: {imbalance_ratio}:1")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(data=df_processed, x="claim_status", ax=axes[0], color="#4c72b0")
axes[0].set_title("Claim status distribution in processed data")
axes[0].set_xlabel("claim_status")
axes[0].set_ylabel("policy_count")

sns.barplot(x=target_dist.index.astype(str), y=target_dist["share_pct"], ax=axes[1], color="#55a868")
axes[1].set_title("Claim status share in processed data")
axes[1].set_xlabel("claim_status")
axes[1].set_ylabel("share_pct")

plt.tight_layout()
plt.show()

In [ ]:
continuous_summary = df_processed[continuous_processed_cols].describe().T.sort_values("std", ascending=False)
outlier_audit = iqr_outlier_audit(df_processed, continuous_processed_cols)

print("Top continuous processed features by standard deviation")
print(continuous_summary[["mean", "std", "min", "25%", "50%", "75%", "max"]].head(15).to_string())
print("\nProcessed continuous-feature outlier audit")
print(outlier_audit.head(15).to_string(index=False))

In [ ]:
top_corr = top_correlations(df_processed, top_n=20)
top_continuous_corr = (
    top_correlations(df_processed[continuous_processed_cols + ["claim_status"]], top_n=min(10, len(continuous_processed_cols)))
)

print("Top processed-feature correlations with claim_status")
print(top_corr.to_string(index=False))
print("\nTop continuous processed-feature correlations with claim_status")
print(top_continuous_corr.to_string(index=False))

plt.figure(figsize=(10, 7))
sns.barplot(data=top_corr, y="feature", x="correlation", color="#8172b3")
plt.title("Top processed-feature correlations with claim_status")
plt.xlabel("correlation")
plt.ylabel("feature")
plt.tight_layout()
plt.show()

In [ ]:
continuous_profile = df_processed.groupby("claim_status")[continuous_processed_cols].mean().T
continuous_profile.columns = ["non_claim_mean", "claim_mean"]
continuous_profile["mean_diff"] = continuous_profile["claim_mean"] - continuous_profile["non_claim_mean"]
continuous_profile["abs_mean_diff"] = continuous_profile["mean_diff"].abs()
continuous_profile = continuous_profile.sort_values("abs_mean_diff", ascending=False)

binary_profile = df_processed.groupby("claim_status")[binary_processed_cols].mean().T
binary_profile.columns = ["non_claim_share", "claim_share"]
binary_profile["pp_diff"] = (binary_profile["claim_share"] - binary_profile["non_claim_share"]) * 100
binary_profile["abs_pp_diff"] = binary_profile["pp_diff"].abs()
binary_profile = binary_profile.sort_values("abs_pp_diff", ascending=False)

print("Top continuous processed features by class mean difference")
print(continuous_profile.head(15).to_string())
print("\nTop encoded / binary processed features by percentage-point difference")
print(binary_profile.head(20).to_string())

In [ ]:
top_continuous_features = continuous_profile.head(6).index.tolist()
plot_sample = df_processed.sample(n=min(6000, len(df_processed)), random_state=42)

fig, axes = plt.subplots(3, 2, figsize=(15, 14))
for ax, col in zip(axes.flat, top_continuous_features):
    sns.boxplot(data=plot_sample, x="claim_status", y=col, ax=ax, color="#9ecae1")
    ax.set_title(f"{col} by claim_status")
    ax.set_xlabel("claim_status")
    ax.set_ylabel(col)

plt.tight_layout()
plt.show()

top_binary_features = binary_profile.head(12).reset_index().rename(columns={"index": "feature"})
plt.figure(figsize=(10, 7))
sns.barplot(data=top_binary_features, y="feature", x="pp_diff", color="#dd8452")
plt.title("Top processed binary / encoded features by class share difference")
plt.xlabel("claim_share - non_claim_share (percentage points)")
plt.ylabel("feature")
plt.tight_layout()
plt.show()

In [ ]:
heatmap_features = top_corr["feature"].head(18).tolist() + ["claim_status"]
corr_subset = df_processed[heatmap_features].corr(numeric_only=True)

plt.figure(figsize=(14, 10))
sns.heatmap(corr_subset, cmap="coolwarm", center=0)
plt.title("Correlation heatmap of top processed features")
plt.tight_layout()
plt.show()

In [ ]:
pca_sample = df_processed.sample(n=min(4000, len(df_processed)), random_state=42)
X_sample = pca_sample[feature_cols]
y_sample = pca_sample["claim_status"].astype(str)

X_scaled = StandardScaler().fit_transform(X_sample)
pca = PCA(n_components=2)
components = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(components, columns=["PC1", "PC2"])
pca_df["claim_status"] = y_sample.values

print(
    f"Explained variance by PC1 and PC2: "
    f"{(pca.explained_variance_ratio_[0] * 100):.2f}% and {(pca.explained_variance_ratio_[1] * 100):.2f}%"
)

plt.figure(figsize=(10, 7))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="claim_status", alpha=0.65, s=35)
plt.title("PCA projection of processed data sample")
plt.tight_layout()
plt.show()

## Notes

- This notebook performs EDA on processed data, so almost every feature is numeric and model-ready.
- One-hot encoding expands categorical variables into many binary indicators, which is why processed duplicate counts can be high even when the raw CSV has no full-row duplicates.
- For modeling, the next safe step is still: split first, then rebalance only on the training data.